# HCLA RQ3 — Revised Explanation Quality Experiment

**Conditions:** Numerical / Feature Attribution / Template / HCLA

**Primary metrics**
1. **Clarity** — 설명의 명료성
2. **Evidence Integration** — 근거를 단순 나열하지 않고 예측 설명으로 통합하는 정도
3. **Calibrated Usefulness** — 과장 없이 모델 판단을 해석하는 데 도움되는 정도

RQ2의 **ESR/UCR/CR/EC**는 중복 측정하지 않습니다. 동일 case와 동일 evidence를 네 방법 모두에 사용합니다.


In [ ]:
# ============================================================
# HCLA RQ3 - Explanation Quality Comparison (Revised)
# Conditions: Numerical / Feature Attribution / Template / HCLA
# Metrics: Clarity / Evidence Integration / Calibrated Usefulness
#
# Design principles
# 1) Same cases, same prediction, same SHAP evidence across all 4 conditions
# 2) Condition is the only manipulated factor
# 3) RQ2 groundedness metrics (ESR/UCR/CR/EC) are NOT duplicated here
# 4) Judge sees explanation + common evidence package, but NOT condition name
# 5) Explicit 1-5 anchors reduce ceiling effects and identical-score collapse
# ============================================================

import os, json, re, time, random, pickle, getpass, warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

In [ ]:
# -----------------------------
# 0. Reproducibility / settings

In [ ]:
# -----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# PILOT: 10 cases -> 40 explanations -> 40 judge calls
# FULL : 50 cases -> 200 explanations -> 200 judge calls
RQ3_PILOT = True
RQ3_PILOT_N = 10
RQ3_FULL_N = 50

# Keep class balance when evidence package contains both classes
BALANCED_SAMPLING = True

# API
QWEN_MODEL = "qwen/qwen3-235b-a22b-2507"
LLAMA_JUDGE_MODEL = "meta-llama/llama-3.3-70b-instruct"
GEN_TEMPERATURE = 0.2
JUDGE_TEMPERATURE = 0.0
QWEN_WORKERS = 5
JUDGE_WORKERS = 5
CHECKPOINT_EVERY = 10

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

In [ ]:
# -----------------------------
# 1. Paths

In [ ]:
# -----------------------------
DEFAULT_BASE = "/content/drive/MyDrive/Colab Notebooks/Results/HCLA_MultiModel_Qwen_Llama_OpenRouter_SHAP"
BASE_DIR = os.environ.get("HCLA_BASE_DIR", DEFAULT_BASE)

# RQ2 creates this file after SHAP evidence extraction.
# You can override via environment variable:
#   os.environ["RQ3_EVIDENCE_PKL"] = "/your/path/evidence_packages.pkl"
DEFAULT_EVIDENCE_CANDIDATES = [
    os.environ.get("RQ3_EVIDENCE_PKL", ""),
    os.path.join(BASE_DIR, "_checkpoints", "evidence_packages.pkl"),
    os.path.join(BASE_DIR, "RQ2", "_checkpoints", "evidence_packages.pkl"),
    os.path.join(BASE_DIR, "rq2", "_checkpoints", "evidence_packages.pkl"),
]

RQ3_OUTPUT_DIR = os.path.join(BASE_DIR, "RQ3_Revised")
CHECKPOINT_DIR = os.path.join(RQ3_OUTPUT_DIR, "_checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RQ3_OUTPUT_DIR, exist_ok=True)

def find_existing(candidates):
    for p in candidates:
        if p and os.path.exists(p):
            return p
    return None

EVIDENCE_PKL = find_existing(DEFAULT_EVIDENCE_CANDIDATES)

if EVIDENCE_PKL is None:
    # lightweight recursive fallback under BASE_DIR
    matches = list(Path(BASE_DIR).rglob("evidence_packages.pkl")) if os.path.exists(BASE_DIR) else []
    if matches:
        EVIDENCE_PKL = str(matches[0])

if EVIDENCE_PKL is None:
    raise FileNotFoundError(
        "evidence_packages.pkl을 찾지 못했습니다.\n"
        "RQ2에서 생성된 evidence_packages.pkl 경로를 "
        "os.environ['RQ3_EVIDENCE_PKL'] 로 지정하세요."
    )

print("✅ Evidence:", EVIDENCE_PKL)
print("✅ Output  :", RQ3_OUTPUT_DIR)

In [ ]:
# -----------------------------
# 2. Load evidence packages

In [ ]:
# -----------------------------
with open(EVIDENCE_PKL, "rb") as f:
    evidence_packages = pickle.load(f)

if not isinstance(evidence_packages, list) or len(evidence_packages) == 0:
    raise ValueError("evidence_packages.pkl 형식이 예상과 다릅니다.")

print("Evidence packages loaded:", len(evidence_packages))
print("Example keys:", evidence_packages[0].keys())

REQUIRED_KEYS = {"case_id", "true_label", "prediction", "evidence"}
missing = REQUIRED_KEYS - set(evidence_packages[0].keys())
if missing:
    raise ValueError(f"Evidence package missing keys: {missing}")

In [ ]:
# -----------------------------
# 3. Select SAME cases for all 4 methods

In [ ]:
# -----------------------------
def balanced_select(packages, n, seed=42):
    rng = np.random.default_rng(seed)
    if not BALANCED_SAMPLING:
        idx = rng.choice(len(packages), size=min(n, len(packages)), replace=False)
        return [packages[i] for i in idx]

    normal = [e for e in packages if int(e.get("true_label", -1)) == 0]
    anomaly = [e for e in packages if int(e.get("true_label", -1)) == 1]

    n0 = n // 2
    n1 = n - n0
    if len(normal) < n0 or len(anomaly) < n1:
        raise ValueError(
            f"Balanced sample impossible: normal={len(normal)}, anomaly={len(anomaly)}, requested={n}"
        )

    idx0 = rng.choice(len(normal), size=n0, replace=False)
    idx1 = rng.choice(len(anomaly), size=n1, replace=False)
    out = [normal[i] for i in idx0] + [anomaly[i] for i in idx1]
    rng.shuffle(out)
    return out

N_CASES = RQ3_PILOT_N if RQ3_PILOT else RQ3_FULL_N
rq3_cases = balanced_select(evidence_packages, N_CASES, SEED)

print(f"RQ3 mode: {'PILOT' if RQ3_PILOT else 'FULL'}")
print("Cases:", len(rq3_cases))
print("Normal:", sum(int(e["true_label"]) == 0 for e in rq3_cases))
print("Anomaly:", sum(int(e["true_label"]) == 1 for e in rq3_cases))
print("Expected explanation conditions:", len(rq3_cases) * 4)

In [ ]:
# -----------------------------
# 4. Helpers

In [ ]:
# -----------------------------
def atomic_csv_save(df, path):
    tmp = path + ".tmp"
    df.to_csv(tmp, index=False, encoding="utf-8-sig")
    os.replace(tmp, path)

def load_csv_or_empty(path, cols):
    if os.path.exists(path):
        try:
            df = pd.read_csv(path)
            for c in cols:
                if c not in df.columns:
                    df[c] = np.nan
            return df[cols]
        except Exception:
            pass
    return pd.DataFrame(columns=cols)

def safe_float(x, ndigits=4):
    try:
        return round(float(x), ndigits)
    except Exception:
        return x

def evidence_rows(e):
    """Normalize evidence rows from RQ2 evidence package."""
    rows = e.get("evidence", [])
    norm = []
    for r in rows:
        norm.append({
            "feature": r.get("feature", r.get("name", "")),
            "value": safe_float(r.get("value", np.nan)),
            "shap": safe_float(r.get("shap", r.get("shap_value", np.nan))),
            "effect": r.get("effect", ""),
        })
    return norm

def probability_text(e):
    p = e.get("anomaly_probability", None)
    if p is None:
        return "not available"
    try:
        return f"{float(p):.4f}"
    except Exception:
        return str(p)

In [ ]:
# -----------------------------
# 5. OpenRouter

In [ ]:
# -----------------------------
try:
    from openai import OpenAI
except ImportError:
    raise ImportError("openai package가 필요합니다: pip install openai")

def load_openrouter_key():
    key = (os.getenv("OPENROUTER_API_KEY") or "").strip()
    if key:
        print("✅ OPENROUTER_API_KEY: environment")
        return key
    try:
        from google.colab import userdata
        key = (userdata.get("OPENROUTER_API_KEY") or "").strip()
        if key:
            print("✅ OPENROUTER_API_KEY: Colab Secret")
            return key
    except Exception:
        pass
    key = getpass.getpass("OpenRouter API Key 입력: ").strip()
    if not key:
        raise ValueError("OpenRouter API Key가 필요합니다.")
    return key

OPENROUTER_API_KEY = load_openrouter_key()

def make_client():
    return OpenAI(
        api_key=OPENROUTER_API_KEY,
        base_url=OPENROUTER_BASE_URL,
        timeout=120.0,
        max_retries=0,
    )

def call_model(model, prompt, temperature=0.0, max_retries=5):
    client = make_client()
    for attempt in range(max_retries):
        try:
            r = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
            )
            text = r.choices[0].message.content
            if not text:
                raise RuntimeError("empty response")
            return text.strip()
        except Exception as ex:
            if attempt >= max_retries - 1:
                raise
            wait = min(60, 5 * (2 ** attempt))
            print(f"[retry {attempt+1}/{max_retries}] {type(ex).__name__}: {ex} -> {wait}s")
            time.sleep(wait)

def extract_json_object(text):
    text = text.strip()
    text = re.sub(r"^```(?:json)?", "", text, flags=re.I).strip()
    text = re.sub(r"```$", "", text).strip()
    try:
        return json.loads(text)
    except Exception:
        m = re.search(r"\{.*\}", text, flags=re.S)
        if not m:
            raise ValueError(text[:1000])
        return json.loads(m.group(0))

In [ ]:
# -----------------------------
# 6. FOUR explanation conditions

In [ ]:
# -----------------------------
# IMPORTANT:
# All four conditions use the SAME prediction/case.
# The difference is explanation representation only.

def make_numerical(e):
    """Score/prediction only. No SHAP attribution."""
    return (
        f"Model prediction: {e['prediction']}. "
        f"Anomaly probability: {probability_text(e)}."
    )

def make_feature_attribution(e):
    """Raw SHAP attribution list. No prose interpretation."""
    rows = evidence_rows(e)
    items = []
    for r in rows:
        items.append(
            f"{r['feature']}={r['value']} (SHAP={r['shap']}, effect={r['effect']})"
        )
    return (
        f"Model prediction: {e['prediction']}; anomaly probability: {probability_text(e)}. "
        "Top feature attributions: " + "; ".join(items) + "."
    )

def make_template(e):
    """Fixed, deterministic template using identical SHAP evidence."""
    rows = evidence_rows(e)
    if not rows:
        return (
            f"The model predicted {e['prediction']} with anomaly probability "
            f"{probability_text(e)}. No feature attribution was available."
        )

    positive = [r for r in rows if str(r["effect"]).lower() in {"increase", "positive", "anomaly", "higher"} or (isinstance(r["shap"], (int,float)) and r["shap"] > 0)]
    negative = [r for r in rows if isinstance(r["shap"], (int,float)) and r["shap"] < 0]

    top = rows[:3]
    top_txt = ", ".join(
        f"{r['feature']} (value={r['value']}, SHAP={r['shap']})" for r in top
    )

    return (
        f"The model predicted {e['prediction']} with anomaly probability "
        f"{probability_text(e)}. The strongest attributed features were {top_txt}. "
        "These feature attributions indicate which inputs most influenced the model output; "
        "they do not by themselves establish a causal or criminal interpretation."
    )

HCLA_PROMPT = """You are the HCLA Expert-Style Justification component.

Generate a concise explanation of the model prediction using ONLY the supplied evidence.
Your explanation must:
1. State the model prediction and, if available, anomaly probability.
2. Integrate the most relevant SHAP evidence into a coherent explanation rather than merely listing features.
3. Distinguish model evidence from real-world causal or criminal conclusions.
4. Avoid unsupported external facts, address history, intent, identity, or causal claims.
5. State uncertainty/limitation when the evidence cannot justify a stronger conclusion.
6. Use 2-4 sentences.

Evidence package:
{evidence}

Return ONLY the explanation text.
"""

def make_hcla(e):
    return call_model(
        QWEN_MODEL,
        HCLA_PROMPT.format(evidence=json.dumps(e, ensure_ascii=False, indent=2)),
        temperature=GEN_TEMPERATURE,
        max_retries=5,
    )

METHODS = ["Numerical", "Feature Attribution", "Template", "HCLA"]

In [ ]:
# -----------------------------
# 7. Generate explanations

In [ ]:
# -----------------------------
GEN_CKPT = os.path.join(CHECKPOINT_DIR, "rq3_explanations.csv")
GEN_COLS = [
    "case_id", "true_label", "prediction", "method",
    "explanation", "evidence_json"
]

gen_df = load_csv_or_empty(GEN_CKPT, GEN_COLS)
rows = gen_df.to_dict("records") if len(gen_df) else []
done = {(str(r["case_id"]), str(r["method"])) for r in rows}

# Deterministic 3 baselines first
for e in rq3_cases:
    for method, fn in [
        ("Numerical", make_numerical),
        ("Feature Attribution", make_feature_attribution),
        ("Template", make_template),
    ]:
        key = (str(e["case_id"]), method)
        if key in done:
            continue
        rows.append({
            "case_id": e["case_id"],
            "true_label": e["true_label"],
            "prediction": e["prediction"],
            "method": method,
            "explanation": fn(e),
            "evidence_json": json.dumps(e, ensure_ascii=False),
        })
        done.add(key)

atomic_csv_save(pd.DataFrame(rows, columns=GEN_COLS), GEN_CKPT)

# HCLA in parallel
hcla_jobs = [e for e in rq3_cases if (str(e["case_id"]), "HCLA") not in done]

def hcla_job(e):
    return e, make_hcla(e)

if hcla_jobs:
    with ThreadPoolExecutor(max_workers=QWEN_WORKERS) as ex:
        futures = [ex.submit(hcla_job, e) for e in hcla_jobs]
        completed = 0
        for fut in as_completed(futures):
            e, explanation = fut.result()
            rows.append({
                "case_id": e["case_id"],
                "true_label": e["true_label"],
                "prediction": e["prediction"],
                "method": "HCLA",
                "explanation": explanation,
                "evidence_json": json.dumps(e, ensure_ascii=False),
            })
            done.add((str(e["case_id"]), "HCLA"))
            completed += 1
            if completed % CHECKPOINT_EVERY == 0:
                atomic_csv_save(pd.DataFrame(rows, columns=GEN_COLS), GEN_CKPT)
            print(f"✅ HCLA generated {completed}/{len(hcla_jobs)}")

explanations_df = pd.DataFrame(rows, columns=GEN_COLS)
explanations_df = explanations_df[
    explanations_df["case_id"].astype(str).isin({str(e["case_id"]) for e in rq3_cases})
].copy()
explanations_df["method"] = pd.Categorical(
    explanations_df["method"], categories=METHODS, ordered=True
)
explanations_df = explanations_df.sort_values(["case_id", "method"]).reset_index(drop=True)
atomic_csv_save(explanations_df, GEN_CKPT)

print("Explanations:", len(explanations_df))
print(explanations_df.groupby("method", observed=False).size())

In [ ]:
# -----------------------------
# 8. Revised RQ3 judge

In [ ]:
# -----------------------------
# Why these metrics?
#
# Clarity:
#   Is the explanation understandable and coherent?
#
# Evidence Integration:
#   Does it meaningfully connect supplied model evidence to the prediction,
#   rather than merely reporting a score or dumping a feature list?
#
# Calibrated Usefulness:
#   Does it help a reviewer interpret the output while maintaining
#   appropriate epistemic caution and avoiding unwarranted conclusions?
#
# RQ2 already measures unsupported/contradicted claims directly.
# Therefore RQ3 does NOT duplicate UCR/ESR/CR/EC.

RQ3_JUDGE = """You are a strict evaluator of AI-generated explanations for an anomaly detector.

Evaluate the explanation ONLY using the supplied common evidence package.
The explanation method name is intentionally hidden.

Score THREE DISTINCT dimensions from 1 to 5.
Do NOT give the same score automatically across dimensions.
Use the anchors literally.

A. CLARITY
1 = confusing, fragmented, or largely uninterpretable
2 = understandable only with substantial effort; major wording/structure problems
3 = generally understandable but somewhat technical, vague, or poorly organized
4 = clear and coherent with only minor issues
5 = exceptionally clear, concise, and easy to understand

B. EVIDENCE_INTEGRATION
1 = gives little/no usable connection between the prediction and supplied evidence
2 = mentions a score or evidence superficially without explaining its relevance
3 = identifies relevant evidence but mostly lists/describes it
4 = clearly connects multiple supplied evidence items to the model output
5 = integrates the strongest supplied evidence into a coherent, specific explanation of the model output

C. CALIBRATED_USEFULNESS
1 = not useful for interpretation, or makes misleading/overconfident conclusions
2 = limited practical interpretive value; important caveats or context are missing
3 = moderately useful; supports basic interpretation but has notable limitations
4 = useful for reviewing the model output and appropriately cautious
5 = highly useful for interpretation, prioritizes what matters, and clearly separates model evidence from unsupported real-world conclusions

IMPORTANT:
- Judge explanation quality, NOT whether the model prediction is correct.
- Do not reward verbosity.
- A feature list may score well on factual specificity but should NOT receive high Evidence Integration unless relationships are actually synthesized.
- A numerical score alone should NOT receive high Evidence Integration.
- Do not infer external AML/criminal facts not present in evidence.
- Use the full 1-5 scale when warranted.
- Give each dimension independently.

Return ONLY valid JSON:
{
  "clarity": 1,
  "evidence_integration": 1,
  "calibrated_usefulness": 1,
  "clarity_reason": "...",
  "evidence_integration_reason": "...",
  "calibrated_usefulness_reason": "..."
}
"""

def judge_one(row):
    evidence = json.loads(row["evidence_json"])
    # condition/method is NOT passed to the judge -> blind evaluation
    prompt = f"""{RQ3_JUDGE}

COMMON EVIDENCE PACKAGE:
{json.dumps(evidence, ensure_ascii=False, indent=2)}

EXPLANATION:
{row['explanation']}
"""
    obj = extract_json_object(
        call_model(
            LLAMA_JUDGE_MODEL,
            prompt,
            temperature=JUDGE_TEMPERATURE,
            max_retries=6,
        )
    )

    out = {
        "case_id": row["case_id"],
        "true_label": row["true_label"],
        "prediction": row["prediction"],
        "method": str(row["method"]),
        "clarity": int(obj["clarity"]),
        "evidence_integration": int(obj["evidence_integration"]),
        "calibrated_usefulness": int(obj["calibrated_usefulness"]),
        "clarity_reason": obj.get("clarity_reason", ""),
        "evidence_integration_reason": obj.get("evidence_integration_reason", ""),
        "calibrated_usefulness_reason": obj.get("calibrated_usefulness_reason", ""),
    }

    for metric in ["clarity", "evidence_integration", "calibrated_usefulness"]:
        if out[metric] < 1 or out[metric] > 5:
            raise ValueError(f"Invalid {metric}: {out[metric]}")
    return out

JUDGE_CKPT = os.path.join(CHECKPOINT_DIR, "rq3_judgments.csv")
JUDGE_COLS = [
    "case_id", "true_label", "prediction", "method",
    "clarity", "evidence_integration", "calibrated_usefulness",
    "clarity_reason", "evidence_integration_reason",
    "calibrated_usefulness_reason",
]

judge_df0 = load_csv_or_empty(JUDGE_CKPT, JUDGE_COLS)
judge_rows = judge_df0.to_dict("records") if len(judge_df0) else []
judge_done = {(str(r["case_id"]), str(r["method"])) for r in judge_rows}

todo = []
for _, row in explanations_df.iterrows():
    key = (str(row["case_id"]), str(row["method"]))
    if key not in judge_done:
        todo.append(row.to_dict())

def judge_job(row):
    return judge_one(row)

if todo:
    with ThreadPoolExecutor(max_workers=JUDGE_WORKERS) as ex:
        futures = [ex.submit(judge_job, row) for row in todo]
        completed = 0
        for fut in as_completed(futures):
            out = fut.result()
            judge_rows.append(out)
            judge_done.add((str(out["case_id"]), out["method"]))
            completed += 1
            if completed % CHECKPOINT_EVERY == 0:
                atomic_csv_save(pd.DataFrame(judge_rows, columns=JUDGE_COLS), JUDGE_CKPT)
            print(f"✅ Judge {completed}/{len(todo)}")

judgments_df = pd.DataFrame(judge_rows, columns=JUDGE_COLS)
judgments_df = judgments_df[
    judgments_df["case_id"].astype(str).isin({str(e["case_id"]) for e in rq3_cases})
].copy()
judgments_df["method"] = pd.Categorical(
    judgments_df["method"], categories=METHODS, ordered=True
)
judgments_df = judgments_df.sort_values(["case_id", "method"]).reset_index(drop=True)
atomic_csv_save(judgments_df, JUDGE_CKPT)

In [ ]:
# -----------------------------
# 9. Summary + discrimination diagnostics

In [ ]:
# -----------------------------
METRICS = ["clarity", "evidence_integration", "calibrated_usefulness"]

summary = judgments_df.groupby("method", observed=False)[METRICS].agg(["mean", "std", "median"])
summary.columns = ["_".join(x) for x in summary.columns]
summary = summary.reset_index()

# Composite score is secondary, not a replacement for three dimensions.
judgments_df["composite"] = judgments_df[METRICS].mean(axis=1)

composite_summary = judgments_df.groupby("method", observed=False)["composite"].agg(
    ["mean", "std", "median"]
).reset_index()

# Ceiling effect (% score == 5): useful diagnostic for the professor's concern.
ceiling_rows = []
for method in METHODS:
    d = judgments_df[judgments_df["method"].astype(str) == method]
    for metric in METRICS:
        ceiling_rows.append({
            "method": method,
            "metric": metric,
            "n": len(d),
            "mean": d[metric].mean(),
            "std": d[metric].std(ddof=1),
            "pct_score_5": 100 * (d[metric] == 5).mean(),
            "unique_scores": int(d[metric].nunique()),
        })
diagnostics_df = pd.DataFrame(ceiling_rows)

# Same-score diagnostic: how often all 3 dimensions receive exactly same score?
judgments_df["all_three_same"] = (
    (judgments_df["clarity"] == judgments_df["evidence_integration"]) &
    (judgments_df["clarity"] == judgments_df["calibrated_usefulness"])
)

same_score_df = judgments_df.groupby("method", observed=False)["all_three_same"].agg(
    n="count", same_n="sum", same_rate="mean"
).reset_index()
same_score_df["same_rate_pct"] = same_score_df["same_rate"] * 100

# Paired wide files for later repeated-measures tests
paired = judgments_df.pivot(
    index="case_id",
    columns="method",
    values=METRICS
)

In [ ]:
# -----------------------------
# 10. Optional repeated-measures statistics

In [ ]:
# -----------------------------
# Friedman test is robust for paired ordinal 1-5 ratings across 4 methods.
stats_rows = []
try:
    from scipy.stats import friedmanchisquare

    for metric in METRICS:
        wide = judgments_df.pivot(index="case_id", columns="method", values=metric)
        wide = wide.dropna()
        if all(m in wide.columns for m in METHODS) and len(wide) >= 3:
            stat, p = friedmanchisquare(*(wide[m].values for m in METHODS))
            stats_rows.append({
                "metric": metric,
                "test": "Friedman",
                "n_cases": len(wide),
                "statistic": stat,
                "p_value": p,
            })
except Exception as ex:
    print("Friedman test skipped:", ex)

stats_df = pd.DataFrame(stats_rows)

In [ ]:
# -----------------------------
# 11. Save

In [ ]:
# -----------------------------
atomic_csv_save(explanations_df, os.path.join(RQ3_OUTPUT_DIR, "rq3_explanations.csv"))
atomic_csv_save(judgments_df, os.path.join(RQ3_OUTPUT_DIR, "rq3_judgments.csv"))
atomic_csv_save(summary, os.path.join(RQ3_OUTPUT_DIR, "rq3_summary.csv"))
atomic_csv_save(composite_summary, os.path.join(RQ3_OUTPUT_DIR, "rq3_composite_summary.csv"))
atomic_csv_save(diagnostics_df, os.path.join(RQ3_OUTPUT_DIR, "rq3_ceiling_diagnostics.csv"))
atomic_csv_save(same_score_df, os.path.join(RQ3_OUTPUT_DIR, "rq3_same_score_diagnostics.csv"))
if len(stats_df):
    atomic_csv_save(stats_df, os.path.join(RQ3_OUTPUT_DIR, "rq3_friedman_test.csv"))

print("\n" + "="*80)
print("RQ3 COMPLETE")
print("="*80)
print("\n[Primary metrics]")
print("1) Clarity               : 설명이 명확하고 이해하기 쉬운가")
print("2) Evidence Integration  : SHAP/모델 근거를 단순 나열이 아니라 설명으로 통합하는가")
print("3) Calibrated Usefulness : 과장 없이 모델 판단을 해석하는 데 실질적으로 도움이 되는가")
print("\n[RQ2와 분리]")
print("Unsupported Claim / ESR / UCR / CR / EC는 RQ2에서 평가하므로 RQ3에서 중복하지 않음.")
print("\n[Methods]")
print("Numerical            = prediction + probability")
print("Feature Attribution  = prediction + probability + raw Top-K SHAP")
print("Template             = 동일 SHAP evidence를 고정 template으로 설명")
print("HCLA                 = 동일 SHAP evidence를 HCLA justification으로 통합 설명")
print("\n[Summary]")
print(summary.round(3).to_string(index=False))
print("\n[Ceiling diagnostics]")
print(diagnostics_df.round(3).to_string(index=False))
print("\n[Same-score diagnostics]")
print(same_score_df.round(3).to_string(index=False))
if len(stats_df):
    print("\n[Friedman]")
    print(stats_df.round(5).to_string(index=False))
print("="*80)